# Day 0 Spike: BITE 3D 강아지 복원 + 치수 추출 테스트

## 목표
1. **Spike 1**: BITE 모델로 강아지 사진 → 3D 메시 복원이 되는지 확인
2. **Spike 2**: 복원된 메시에서 신체 치수 추출이 가능한지 확인

## Pass/Fail 기준
- Spike 1 Pass: 강아지 형상이 인식 가능한 3D 메시 생성
- Spike 2 Pass: 주요 치수(가슴둘레, 등길이) 오차 3cm 이내가 10마리 중 7마리 이상

## 환경
- Google Colab (GPU 런타임) 또는 로컬 GPU 데스크탑
- Python 3.7+ (BITE 호환)
- PyTorch + CUDA

## Step 0: 환경 확인

In [ ]:
import torch
print(f"Python: {__import__('sys').version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Step 1: BITE 코드 클론 + 의존성 설치

BITE HuggingFace Space에서 전체 코드와 가중치를 다운로드합니다.

In [ ]:
import os

# Colab인 경우 Google Drive 마운트 (선택)
try:
    from google.colab import drive
    IN_COLAB = True
    # drive.mount('/content/drive')  # 필요시 주석 해제
except ImportError:
    IN_COLAB = False

WORK_DIR = '/content' if IN_COLAB else os.path.expanduser('~/projects/dog-measurement')
BITE_DIR = os.path.join(WORK_DIR, 'bite_release')
print(f"Working directory: {WORK_DIR}")
print(f"BITE directory: {BITE_DIR}")

In [ ]:
%%bash
# BITE HuggingFace Space 클론 (코드 + 모델 가중치 포함)
if [ ! -d "bite_release" ]; then
    git lfs install
    git clone https://huggingface.co/spaces/runa91/bite_gradio bite_release
    echo "BITE 클론 완료"
else
    echo "BITE 이미 존재"
fi

# 디렉토리 구조 확인
ls -la bite_release/
echo "---"
ls bite_release/checkpoint/ 2>/dev/null || echo "checkpoint 폴더 없음"
echo "---"
ls bite_release/data/ 2>/dev/null || echo "data 폴더 없음"

In [ ]:
# 의존성 설치
# 주의: BITE는 구버전 PyTorch(1.6)를 요구하지만,
# 최신 PyTorch에서도 핵심 기능은 작동할 수 있음

!pip install trimesh scipy matplotlib opencv-python-headless chumpy openpyxl dominate
!pip install kornia

# pytorch3d 설치 (버전에 따라 다름)
import sys
import torch
pyt_version_str = torch.__version__.split('+')[0].replace('.', '')
version_str = ''.join([f'py3{sys.version_info.minor}_cu', 
                       torch.version.cuda.replace('.',''), 
                       f'_pyt{pyt_version_str}'])
print(f"pytorch3d version string: {version_str}")

# Colab의 경우 pre-built wheel 사용
if IN_COLAB:
    !pip install --no-index --no-cache-dir pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/{version_str}/download.html
else:
    # 로컬: conda install 또는 소스 빌드
    print("로컬 환경에서는 pytorch3d를 별도로 설치해야 합니다.")
    print("conda install -c fvcore -c iopath -c conda-forge fvcore iopath")
    print("pip install 'git+https://github.com/facebookresearch/pytorch3d.git'")

# FrEIA (BITE 의존성)
!pip install git+https://github.com/runa91/FrEIA.git

## Step 2: BITE 모델 로드 테스트

모델이 로드되는지 확인합니다. 이 단계가 성공하면 Spike 1의 절반은 완료.

In [ ]:
import sys
sys.path.insert(0, BITE_DIR)
sys.path.insert(0, os.path.join(BITE_DIR, 'src'))

# SMAL 모델 로드 테스트
try:
    from src.smal_pytorch.smal_model.smal_torch_new import SMAL
    print("[OK] SMAL 모듈 import 성공")
except Exception as e:
    print(f"[FAIL] SMAL import 실패: {e}")
    print("→ BITE 코드의 Python/PyTorch 호환성 문제. 에러 메시지를 확인하세요.")

In [ ]:
# SMAL 모델 인스턴스 생성
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

try:
    smal = SMAL(
        smal_model_type='39dogs_norm',
        template_name='neutral',
        logscale_part_list=['legs_l', 'legs_f', 'tail_l', 'tail_f', 
                           'ears_y', 'ears_l', 'head_l']
    ).to(device)
    print(f"[OK] SMAL 모델 로드 성공")
    print(f"  - Vertices: {smal.v_template.shape}")
    print(f"  - Faces: {smal.f.shape}")
except Exception as e:
    print(f"[FAIL] SMAL 모델 로드 실패: {e}")

## Step 3: SMAL 메시 토폴로지 탐색 (랜드마크 발견)

SMAL 메시의 vertex가 강아지 몸의 어디에 해당하는지 시각화합니다.
이 정보가 치수 추출의 기반이 됩니다.

In [ ]:
import numpy as np
import trimesh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# SMAL 기본 메시 (neutral pose, neutral shape) 추출
with torch.no_grad():
    betas = torch.zeros(1, 30).to(device)  # neutral shape
    betas_limbs = torch.zeros(1, 7).to(device)
    pose = torch.zeros(1, 34*3).to(device)  # neutral pose
    trans = torch.zeros(1, 3).to(device)
    
    verts, keyp_3d, _ = smal(
        beta=betas, betas_limbs=betas_limbs,
        pose=pose, trans=trans,
        keyp_conf='olive', get_skin=True
    )

verts_np = verts[0].cpu().numpy()
faces_np = smal.f.cpu().numpy() if torch.is_tensor(smal.f) else smal.f

print(f"Vertices shape: {verts_np.shape}")
print(f"Faces shape: {faces_np.shape}")
print(f"Bounding box:")
print(f"  X: [{verts_np[:,0].min():.3f}, {verts_np[:,0].max():.3f}]")
print(f"  Y: [{verts_np[:,1].min():.3f}, {verts_np[:,1].max():.3f}]")
print(f"  Z: [{verts_np[:,2].min():.3f}, {verts_np[:,2].max():.3f}]")

dims = verts_np.max(axis=0) - verts_np.min(axis=0)
print(f"\nDimensions (SMAL units):")
print(f"  Length: {dims[0]:.3f}")
print(f"  Height: {dims[1]:.3f}")
print(f"  Width:  {dims[2]:.3f}")

In [ ]:
# 3D 시각화
fig = plt.figure(figsize=(16, 5))

views = [
    ("Side view", 0, 0),
    ("Front view", 0, 90),
    ("Top view", 90, 0),
]

for i, (title, elev, azim) in enumerate(views):
    ax = fig.add_subplot(1, 3, i+1, projection='3d')
    ax.scatter(verts_np[::5, 0], verts_np[::5, 1], verts_np[::5, 2], 
              c=verts_np[::5, 0], cmap='viridis', s=1, alpha=0.5)
    ax.set_title(title)
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

plt.tight_layout()
plt.savefig('smal_neutral_mesh.png', dpi=150)
plt.show()
print("메시 시각화 완료 — 강아지 형상이 보이면 Spike 1 부분 성공")

In [ ]:
# SMAL body parts 시각화 (vertex별 파트 레이블)
# SMAL 모델에 파트 레이블이 있는지 확인
if hasattr(smal, 'parts'):
    parts = smal.parts
    if torch.is_tensor(parts):
        parts = parts.cpu().numpy()
    print(f"Body parts available: {np.unique(parts)}")
    
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    scatter = ax.scatter(verts_np[:, 0], verts_np[:, 1], verts_np[:, 2],
                        c=parts, cmap='tab20', s=2, alpha=0.7)
    plt.colorbar(scatter, label='Body Part ID')
    ax.set_title('SMAL Body Parts')
    ax.view_init(elev=0, azim=0)
    plt.savefig('smal_body_parts.png', dpi=150)
    plt.show()
else:
    print("SMAL 모델에 parts 속성 없음 — skinning weights에서 유추 필요")
    if hasattr(smal, 'W'):
        W = smal.W
        if torch.is_tensor(W):
            W = W.cpu().numpy()
        parts_from_weights = np.argmax(W, axis=1)
        print(f"Skinning weights → parts: {np.unique(parts_from_weights)}")
        
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        scatter = ax.scatter(verts_np[:, 0], verts_np[:, 1], verts_np[:, 2],
                            c=parts_from_weights, cmap='tab20', s=2, alpha=0.7)
        plt.colorbar(scatter, label='Joint (max weight)')
        ax.set_title('SMAL Parts from Skinning Weights')
        ax.view_init(elev=0, azim=0)
        plt.savefig('smal_body_parts.png', dpi=150)
        plt.show()

## Step 4: 강아지 사진으로 BITE 추론 실행

실제 강아지 사진을 넣어서 3D 복원이 되는지 확인합니다.

In [ ]:
# 테스트 이미지 준비
# BITE에 포함된 예제 이미지 사용
import glob

test_images = glob.glob(os.path.join(BITE_DIR, 'datasets', 'test_image_crops', '*'))
if test_images:
    print(f"예제 이미지 {len(test_images)}장 발견:")
    for img_path in test_images[:5]:
        print(f"  - {os.path.basename(img_path)}")
else:
    print("예제 이미지 없음 — 직접 업로드하세요")
    if IN_COLAB:
        from google.colab import files
        uploaded = files.upload()
        test_images = list(uploaded.keys())
        print(f"업로드된 이미지: {test_images}")

In [ ]:
# BITE 전체 추론 파이프라인 실행
# gradio_demo.py의 로직을 단순화한 버전

from PIL import Image
import torchvision

# 테스트 이미지 로드
test_img_path = test_images[0] if test_images else None
if test_img_path is None:
    print("테스트 이미지가 없습니다. 업로드 후 다시 실행하세요.")
else:
    img = Image.open(test_img_path).convert('RGB')
    print(f"이미지 로드: {test_img_path}")
    print(f"  크기: {img.size}")
    plt.imshow(img)
    plt.title('Input Image')
    plt.axis('off')
    plt.show()

In [ ]:
# BITE 추론 실행
# 이 셀은 BITE 코드 호환성에 따라 수정이 필요할 수 있음

try:
    from scripts.gradio_demo import run_bite_inference, run_bbox_inference
    
    # 1. Bounding box 검출
    print("[1/3] Bounding box 검출 중...")
    bbox_img, bbox = run_bbox_inference(test_img_path)
    print(f"  bbox: {bbox}")
    
    # 2. BITE 추론 (test-time optimization 포함)
    print("[2/3] BITE 3D 복원 중... (약 2-5분 소요)")
    result = run_bite_inference(
        test_img_path, 
        bbox=bbox, 
        apply_ttopt=True,
        dog_name="spike_test"
    )
    print("[3/3] 완료!")
    
    # 결과 확인
    if result:
        print(f"\n[SPIKE 1 PASS] 3D 메시 생성 성공!")
        print(f"  결과 파일: {result}")
    else:
        print("\n[SPIKE 1 FAIL] 3D 메시 생성 실패")

except ImportError as e:
    print(f"[호환성 문제] BITE 코드 import 실패: {e}")
    print("→ PyTorch/Python 버전 호환성 문제. 아래 대안 셀을 실행하세요.")
except Exception as e:
    print(f"[실행 오류] {e}")
    import traceback
    traceback.print_exc()

## Step 5: 복원된 메시에서 치수 추출 (Spike 2)

BITE가 생성한 3D 메시에서 실제 치수를 추출합니다.

In [ ]:
# 생성된 메시 로드
result_mesh_path = None  # Step 4에서 생성된 경로로 업데이트

# GLB 또는 OBJ 파일에서 메시 로드
if result_mesh_path and os.path.exists(result_mesh_path):
    mesh = trimesh.load(result_mesh_path)
    result_verts = np.array(mesh.vertices)
    result_faces = np.array(mesh.faces)
    print(f"메시 로드 완료: {result_verts.shape[0]} vertices, {result_faces.shape[0]} faces")
else:
    print("메시 파일 경로를 설정하세요.")
    print("또는 Step 3의 neutral 메시로 테스트합니다.")
    result_verts = verts_np
    result_faces = faces_np

In [ ]:
# 치수 추출 (bounding box 기반 — 랜드마크 없이도 대략적 치수)

# dog-measurement 프로젝트의 extractor 사용
# Colab에서는 직접 코드를 여기에 포함

dims = result_verts.max(axis=0) - result_verts.min(axis=0)

# SMAL 단위 → 실제 단위 변환
# SMAL 모델의 기본 단위는 미터 단위에 근접 (확인 필요)
# neutral 모델의 body length가 약 0.5-0.8m이면 적절
smal_body_length = dims[0]  # X축 = 길이 방향 (확인 필요)
print(f"\nSMAL 단위 치수:")
print(f"  길이(X): {dims[0]:.4f}")
print(f"  높이(Y): {dims[1]:.4f}")
print(f"  너비(Z): {dims[2]:.4f}")

# 만약 SMAL 단위가 미터라면:
scale_to_cm = 100.0
print(f"\n추정 실제 치수 (SMAL=미터 가정):")
print(f"  체장(body length): {dims[0] * scale_to_cm:.1f} cm")
print(f"  체고(body height): {dims[1] * scale_to_cm:.1f} cm")
print(f"  체폭(body width):  {dims[2] * scale_to_cm:.1f} cm")
print(f"  추정 가슴둘레:     {(dims[1] + dims[2]) * scale_to_cm * 1.1:.1f} cm")

print(f"\n⚠️ 이 값은 bounding box 기반 추정치입니다.")
print(f"   정확한 둘레 측정은 mesh cross-section 방식으로 해야 합니다.")
print(f"   또한 SMAL 단위가 실제로 미터인지 확인이 필요합니다.")

In [ ]:
# Cross-section 기반 둘레 측정 (extractor.py 로직 인라인)

def mesh_cross_section(vertices, faces, plane_point, plane_normal):
    """메시를 평면으로 절단하여 교차점 반환."""
    plane_normal = plane_normal / np.linalg.norm(plane_normal)
    signed_dists = np.dot(vertices - plane_point, plane_normal)
    points = []
    for face in faces:
        for i, j in [(face[0], face[1]), (face[1], face[2]), (face[2], face[0])]:
            if signed_dists[i] * signed_dists[j] < 0:
                t = signed_dists[i] / (signed_dists[i] - signed_dists[j])
                points.append(vertices[i] + t * (vertices[j] - vertices[i]))
    if len(points) < 3:
        return None
    pts = np.array(points)
    # 각도순 정렬
    centroid = pts.mean(axis=0)
    rel = pts - centroid
    ref = np.array([0, 1, 0]) if abs(plane_normal[1]) < 0.9 else np.array([1, 0, 0])
    u = np.cross(plane_normal, ref)
    u /= np.linalg.norm(u)
    v = np.cross(plane_normal, u)
    angles = np.arctan2(np.dot(rel, v), np.dot(rel, u))
    return pts[np.argsort(angles)]

def circumference(points):
    """정렬된 폐합 곡선의 둘레."""
    n = len(points)
    return sum(np.linalg.norm(points[(i+1)%n] - points[i]) for i in range(n))

# 가슴 위치에서 절단 (X축 중간 부근)
x_range = result_verts[:, 0].max() - result_verts[:, 0].min()
chest_x = result_verts[:, 0].min() + x_range * 0.35  # 앞쪽 35% 지점 (가슴 추정)
belly_x = result_verts[:, 0].min() + x_range * 0.55  # 중간 55% 지점 (배 추정)
neck_x = result_verts[:, 0].min() + x_range * 0.2    # 앞쪽 20% 지점 (목 추정)

normal = np.array([1.0, 0.0, 0.0])  # X축 수직 절단

for name, x_pos in [("목", neck_x), ("가슴", chest_x), ("배", belly_x)]:
    plane_point = np.array([x_pos, 0, 0])
    cs = mesh_cross_section(result_verts, result_faces, plane_point, normal)
    if cs is not None:
        circ = circumference(cs) * scale_to_cm
        print(f"{name} 둘레 (cross-section): {circ:.1f} cm  (교차점 {len(cs)}개)")
    else:
        print(f"{name}: 교차점 부족")

## Step 6: 결과 정리 + Pass/Fail 판정

In [ ]:
# === Day 0 Spike 결과 정리 ===

print("=" * 60)
print("Day 0 Spike 결과")
print("=" * 60)

print("\n[Spike 1: 3D 형상 복원]")
print("  SMAL 모델 로드: ___")
print("  강아지 사진 → 3D 메시 복원: ___")
print("  시각적 품질: ___")
print("  판정: ___ (PASS / FAIL)")

print("\n[Spike 2: 치수 추출]")
print("  Cross-section 기반 둘레 계산: ___")
print("  절대 스케일 변환: ___")
print("  줄자 대비 오차: ___ cm")
print("  판정: ___ (PASS / FAIL)")

print("\n[종합 판정]")
print("  ___ (PASS → 본 개발 진행 / FAIL → Approach A 피벗)")
print("\n⚠️ 위 ___ 부분을 실제 결과로 채워주세요.")